# पाठ 10 - उत्पादन में AI एजेंट

In this lesson you will learn **production patterns** for AI agents using the Microsoft Agent Framework (Python). We cover:

- **ऑब्ज़रवेबिलिटी** — एजेंट इंटरैक्शन में समय मापन और लॉगिंग जोड़ना
- **मूल्यांकन** — एक मूल्यांकन एजेंट का उपयोग करके प्रतिक्रिया गुणवत्ता को स्कोर करना
- **लागत प्रबंधन** — टोकन अनुकूलन और मॉडल चयन के लिए रणनीतियाँ

The scenario is a **यात्रा एजेंट** that helps users plan trips, with monitoring and evaluation layered on top.


## सेटअप


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## प्रोडक्शन पर विचार

AI एजेंट्स को प्रोटोटाइप से प्रोडक्शन में ले जाने के लिए तीन स्तंभों पर सावधानीपूर्वक ध्यान देने की आवश्यकता होती है:

1. **निगरानी** — आपको यह दिखाई देना चाहिए कि एजेंट क्या कर रहा है, उसे कितना समय लग रहा है, और वह किन टूल्स को कॉल कर रहा है। ट्रेसिंग और लॉगिंग के बिना, प्रोडक्शन समस्याओं का डिबग करना लगभग असंभव है।

2. **मूल्यांकन** — स्वचालित गुणवत्ता जांच यह सुनिश्चित करती है कि एजेंट के उत्तर समय के साथ सटीक, पूर्ण और सहायक बने रहें। एक मूल्यांकन एजेंट परिभाषित मानदंडों के खिलाफ उत्तरों को स्कोर कर सकता है।

3. **लागत प्रबंधन** — टोकन उपयोग सीधे लागत को प्रभावित करता है। प्रॉम्प्ट अनुकूलन, मॉडल चयन, और कैशिंग जैसी रणनीतियाँ बिना गुणवत्ता से समझौता किए खर्चों को नियंत्रण में रखने में मदद करती हैं।


## निगरानी योग्य एजेंट बनाना

हम यात्रा उपकरणों को परिभाषित करते हैं और विलंबता की निगरानी के लिए एजेंट कॉल के चारों ओर समय मापन लगाते हैं। प्रोडक्शन में आप OpenTelemetry या इसी तरह के किसी ट्रेसिंग बैकएंड के साथ एकीकृत करेंगे।


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## मूल्यांकन पैटर्न

एक सामान्य प्रोडक्शन पैटर्न यह है कि दूसरे एजेंट का उपयोग **मूल्यांकनकर्ता** के रूप में किया जाए। मूल्यांकनकर्ता प्राथमिक एजेंट की प्रतिक्रिया को पूर्वनिर्धारित मानदंडों जैसे पूर्णता, सटीकता और सहायकता के आधार पर स्कोर करता है।

यह सक्षम बनाता है:
- उपयोगकर्ताओं तक पहुँचने से पहले प्रतिक्रियाओं के लिए स्वचालित गुणवत्ता जाँच
- प्रॉम्प्ट या मॉडल बदलने पर रिग्रेशन का पता लगाना
- समय के साथ एजेंट के प्रदर्शन की निरंतर निगरानी


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## लागत प्रबंधन रणनीतियाँ

लागतों को नियंत्रित करना उत्पादन AI एजेंट्स के लिए अत्यंत महत्वपूर्ण है। यहाँ प्रमुख रणनीतियाँ दी गई हैं:

| Strategy | Description |
|---|---|
| **प्रॉम्प्ट अनुकूलन** | सिस्टम निर्देशों को संक्षिप्त रखें। इनपुट टोकन कम करने के लिए अनावश्यक संदर्भ हटाएँ। |
| **मॉडल चयन** | साधारण कार्यों (जैसे वर्गीकरण या निष्कर्षण) के लिए छोटे, सस्ते मॉडल (उदा. GPT-4o-mini) का उपयोग करें, और जटिल तर्क के लिए बड़े मॉडल सुरक्षित रखें। |
| **कैशिंग** | टूल परिणामों और बार-बार पूछताछों को कैश करें ताकि अनावश्यक API कॉल्स से बचा जा सके। |
| **टोकन बजट** | अनपेक्षित रूप से लंबे उत्तरों को रोकने के लिए `max_tokens` सीमाएँ सेट करें। |
| **बैचिंग** | जहाँ संभव हो, कई उपयोगकर्ता प्रश्नों को एकल API कॉल में समूहित करें। |

व्यवहार में, एक स्तरीय दृष्टिकोण अच्छा काम करता है: सीधे-सादे अनुरोधों को तेज़, सस्ते मॉडल पर भेजें और केवल जटिल प्रश्नों को अधिक सक्षम मॉडल तक बढ़ाएँ।


## सारांश

इस पाठ में आपने यह सीखा कि:

1. **अवलोकनीयता जोड़ें** एजेंट इंटरैक्शन में समय मापन और लॉगिंग के साथ, ट्रेसिंग और मॉनिटरिंग के लिए आधार तैयार करना।
2. **एजेंट प्रतिक्रियाओं का मूल्यांकन करें** स्वचालित रूप से एक मूल्यांकन एजेंट का उपयोग करके जो पूर्णता, सटीकता, और उपयोगिता को स्कोर करता है।
3. **लागत प्रबंधित करें** प्रॉम्प्ट अनुकूलन, मॉडल चयन, कैशिंग, और टोकन बजट के माध्यम से।

ये प्रोडक्शन पैटर्न सुनिश्चित करने में मदद करते हैं कि आपके AI एजेंट बड़े पैमाने पर विश्वसनीय, मापनीय, और लागत-प्रभावी हों।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
अस्वीकरण:
यह दस्तावेज़ AI अनुवाद सेवा Co-op Translator (https://github.com/Azure/co-op-translator) का उपयोग करके अनुवादित किया गया है। हम सटीकता के लिए प्रयासरत हैं, फिर भी कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ को उसकी मूल भाषा में अधिकारिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी प्रकार की गलतफ़हमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
